# Data Description

- Author: Codex
- Date first created: 20261013
- Background: 這是原始假訂單資料清理後的版本。原本有 122 列，刪除 2 列重複紀錄、3 筆件數缺值和 2 筆負單價後，剩下 115 筆。再合併商品類別表與日期表，加入類別名稱、星期及平日／週末資訊，共 9 欄。兩次合併皆完整配對，筆數沒有增加。
- Unit of observation: 每列是一筆訂單，`order_id` 不重複。
- Sample period: 2026-09-01 至 2026-09-30。
- Known issues: 資料為人工生成，只用來練習整理與分析流程，不能用來描述真實消費行為。
- Definition for each variable:
    - `order_id`：訂單編號。
    - `order_date`：訂單日期，格式為 YYYY-MM-DD。
    - `category`：商品類別，已統一為 A、B、C，沒有前後空白。
    - `quantity`：購買件數，保留的資料皆為 1 至 5 件的整數。
    - `unit_price_twd`：每件商品的單價，單位為新臺幣元，負值已排除。
    - `order_amount_twd`：訂單金額，等於購買件數乘以單價，單位為新臺幣元。
    - `category_name`：商品類別名稱，A＝飲品、B＝點心、C＝輕食。
    - `weekday`：訂單日期對應的星期。
    - `day_type`：星期六、日記為週末，其餘為平日；不另列國定假日。
- Data path: [data/processed/demo_orders_clean.csv](demo_orders_clean.csv)
- Source data: [原始 CSV](../raw/demo_orders.csv)；[原始資料說明](../raw/demo_orders_documentation.ipynb)
- Source lookup tables: [商品類別表](../raw/demo_categories.csv)（[說明](../raw/demo_categories_documentation.ipynb)）、[日期表](../raw/demo_calendar.csv)（[說明](../raw/demo_calendar_documentation.ipynb)）。
- Processing code: [data/code/demo_orders_processing.ipynb](../code/demo_orders_processing.ipynb)

要重建資料，先在專案根目錄執行 `python data/code/demo_generate_orders.py`，再執行上方的處理 notebook。

接下來在 [分析 notebook](../../finding/demo_orders_analysis.ipynb) 查看分布與圖表。

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 找到專案資料夾，從不同位置開啟 notebook 都能讀到資料。
project_root = Path.cwd().resolve()
while not ((project_root / "AGENTS.md").is_file() and (project_root / "templates").is_dir()):
    if project_root == project_root.parent:
        raise FileNotFoundError("請從專案資料夾內開啟 notebook。")
    project_root = project_root.parent


## Change for Your Own Data

讀取前一步合併好的資料。如果要調整清理或合併方式，回到 data/code 的處理 notebook 修改，再重新執行。

In [2]:
data_path = project_root / "data/processed/demo_orders_clean.csv"

## Summary

三表合併後共有 115 筆、9 個欄位。訂單仍是一列一筆，新增的三個欄位來自商品類別與日期表。

In [3]:
df = pd.read_csv(data_path)
print(f"整理後共有 {len(df)} 筆、{len(df.columns)} 欄。")


整理後共有 115 筆、9 欄。


### Sample

以下是前 10 筆整理後的訂單。例如第一筆買了 1 件、單價 80 元，訂單金額為 80 元；第二筆買了 3 件、單價 97 元，金額為 291 元。

In [4]:
df.head(10)

,order_id,order_date,category,quantity,unit_price_twd,order_amount_twd,category_name,weekday,day_type
0,D0001,2026-09-01,A,1,80,80,飲品,星期二,平日
1,D0002,2026-09-02,B,3,97,291,點心,星期三,平日
2,D0003,2026-09-03,C,5,114,570,輕食,星期四,平日
3,D0004,2026-09-04,A,2,131,262,飲品,星期五,平日
4,D0005,2026-09-05,B,4,148,592,點心,星期六,週末
5,D0006,2026-09-06,C,1,165,165,輕食,星期日,週末
6,D0007,2026-09-07,A,3,182,546,飲品,星期一,平日
7,D0008,2026-09-08,B,5,199,995,點心,星期二,平日
8,D0010,2026-09-10,A,4,233,932,飲品,星期四,平日
9,D0011,2026-09-11,B,1,250,250,點心,星期五,平日


### Summary Stats

115 筆訂單的平均購買件數約為 3.01 件，平均金額約為 539.75 元。訂單金額從 80 元到 1,420 元不等，所有欄位都沒有缺值。

In [5]:
print(f"整理後資料摘要（N={len(df)}）")
display(df.describe(include="all"))

column_info = pd.DataFrame({
    "型態": df.dtypes.astype(str),
    "缺值數": df.isna().sum(),
})
print("欄位型態與缺值數")
display(column_info)

整理後資料摘要（N=115）


,order_id,order_date,category,quantity,unit_price_twd,order_amount_twd,category_name,weekday,day_type
count,115,115,115,115.000000,115.000000,115.000000,115,115,115
unique,115,30,3,NaN,NaN,NaN,3,7,2
top,D0001,2026-09-01,A,NaN,NaN,NaN,飲品,星期二,平日
freq,1,4,39,NaN,NaN,NaN,39,19,86
mean,NaN,NaN,NaN,3.008696,179.191304,539.747826,NaN,NaN,NaN
std,NaN,NaN,NaN,1.435732,64.829181,337.452553,NaN,NaN,NaN
min,NaN,NaN,NaN,1.000000,80.000000,80.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,2.000000,122.500000,262.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,3.000000,182.000000,485.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,4.000000,233.000000,750.000000,NaN,NaN,NaN


欄位型態與缺值數


,型態,缺值數
order_id,object,0
order_date,object,0
category,object,0
quantity,int64,0
unit_price_twd,int64,0
order_amount_twd,int64,0
category_name,object,0
weekday,object,0
day_type,object,0


### Assertions

最後確認沒有重複訂單或缺值，再檢查金額、商品名稱與日期是否正確。

In [6]:
assert df["order_id"].is_unique, "訂單編號有重複。"
assert not df.isna().any().any(), "合併後仍有缺值。"
assert (df["quantity"] > 0).all()
assert (df["quantity"] % 1 == 0).all()
assert (df["unit_price_twd"] > 0).all()
assert (df["order_amount_twd"] == df["quantity"] * df["unit_price_twd"]).all()

category_names = {"A": "飲品", "B": "點心", "C": "輕食"}
assert df["category_name"].equals(df["category"].map(category_names))
dates = pd.to_datetime(df["order_date"])
assert df["day_type"].eq("週末").equals(dates.dt.dayofweek.ge(5))
weekday_names = ["星期一", "星期二", "星期三", "星期四", "星期五", "星期六", "星期日"]
assert df["weekday"].equals(dates.dt.dayofweek.map(dict(enumerate(weekday_names))))
print("訂單沒有重複或缺值，金額、商品名稱和日期資訊都正確。")


訂單沒有重複或缺值，金額、商品名稱和日期資訊都正確。
